# BM25 어휘 검색 (BM25 Lexical Search)

**Skilljar Lesson L06 대응**

이 노트북에서 다루는 내용:
1. Semantic vs Lexical 검색 비교
2. BM25 알고리즘 원리
3. BM25Index 클래스 구현
4. 두 검색 방식의 강점/약점 비교 실험

In [ ]:
# ── Setup ──────────────────────────────────────────────
import math
from collections import Counter

## §1. 왜 BM25가 필요한가?

벡터 검색은 **의미적 유사성**에 강하지만, **정확한 키워드**에는 약할 수 있습니다.

| 쿼리 | Semantic Search | BM25 |
|-------|----------------|------|
| "건물 내력" | "구조물 강도" 검색 가능 | 정확한 단어 필요 |
| "KDS 41 10 20" | 숫자 의미 파악 어려움 | 정확히 매칭 |
| "fck 30MPa" | 부분적 이해 | 정확히 매칭 |

## §2. BM25Index 클래스 구현

In [ ]:
class BM25Index:
    """BM25 기반 어휘 검색 인덱스"""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.documents = []
        self.doc_lengths = []
        self.avg_doc_length = 0
        self.doc_freqs = Counter()
        self.doc_term_freqs = []

    def _tokenize(self, text: str) -> list[str]:
        """간단한 토크나이저 — 소문자 변환 후 공백 분할"""
        return text.lower().split()

    def add_documents(self, documents: list[str]):
        """문서 청크를 인덱스에 추가한다."""
        self.documents = documents

        for doc in documents:
            tokens = self._tokenize(doc)
            self.doc_lengths.append(len(tokens))

            term_freq = Counter(tokens)
            self.doc_term_freqs.append(term_freq)

            for term in set(tokens):
                self.doc_freqs[term] += 1

        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths)
        print(f"\u2705 {len(documents)}개 문서 BM25 인덱싱 완료")

    def _idf(self, term: str) -> float:
        """역문서 빈도 (IDF) 계산"""
        n = len(self.documents)
        df = self.doc_freqs.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """BM25 점수로 문서를 검색한다."""
        query_tokens = self._tokenize(query)
        scores = []

        for i, doc in enumerate(self.documents):
            score = 0
            for term in query_tokens:
                tf = self.doc_term_freqs[i].get(term, 0)
                idf = self._idf(term)
                dl = self.doc_lengths[i]

                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (
                    1 - self.b + self.b * dl / self.avg_doc_length
                )
                score += idf * numerator / denominator

            scores.append(score)

        top_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )[:top_k]

        return [
            {
                "text": self.documents[i],
                "score": scores[i],
                "id": f"doc_{i}"
            }
            for i in top_indices
        ]

## §3. BM25 검색 테스트

In [ ]:
# 샘플 청크
sample_chunks = [
    "KDS 41 10 20: 이 기준은 건축물의 구조안전성을 확보하기 위한 최소한의 요구사항을 정한다.",
    "고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3이다.",
    "콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다.",
    "RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다.",
    "기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이다.",
    "적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 2.0 kN/m2, 사무실 2.5 kN/m2이다.",
]

bm25_index = BM25Index()
bm25_index.add_documents(sample_chunks)

In [ ]:
# BM25 검색 테스트
queries = [
    "KDS 41 10 20",          # 정확한 조항 번호
    "fck 40 MPa",            # 기술 용어
    "최소 철근비",             # 일반 키워드
    "사무실 적재하중",         # 복합 키워드
]

for query in queries:
    results = bm25_index.search(query, top_k=2)
    print(f"\n쿼리: '{query}'")
    for i, r in enumerate(results, 1):
        print(f"  {i}위 [{r['score']:.4f}] {r['text'][:80]}...")

## §4. BM25 알고리즘 상세 분석

BM25 점수 계산 과정을 단계별로 살펴봅니다.

In [ ]:
# BM25 점수 분해 분석
query = "최소 철근비"
query_tokens = bm25_index._tokenize(query)

print(f"쿼리: '{query}'")
print(f"토큰: {query_tokens}")
print(f"평균 문서 길이: {bm25_index.avg_doc_length:.1f}\n")

for i, doc in enumerate(sample_chunks):
    total_score = 0
    details = []
    for term in query_tokens:
        tf = bm25_index.doc_term_freqs[i].get(term, 0)
        idf = bm25_index._idf(term)
        dl = bm25_index.doc_lengths[i]
        k1, b = bm25_index.k1, bm25_index.b
        avgdl = bm25_index.avg_doc_length

        num = tf * (k1 + 1)
        den = tf + k1 * (1 - b + b * dl / avgdl)
        term_score = idf * num / den if den > 0 else 0
        total_score += term_score
        if tf > 0:
            details.append(f"'{term}'(tf={tf}, idf={idf:.2f})={term_score:.4f}")

    if total_score > 0:
        print(f"  문서 {i}: score={total_score:.4f}")
        print(f"    분해: {' + '.join(details)}")
        print(f"    텍스트: {doc[:60]}...")

## 정리

- BM25는 **키워드 빈도** 기반 검색으로, 정확한 용어/번호 매칭에 강하다
- TF (단어 빈도) + IDF (역문서 빈도) + 문서 길이 정규화를 결합
- 벡터 검색과 **상호보완적** — 각각의 강점이 다르다

다음 노트북에서는 두 검색을 **하이브리드로 결합**합니다. → `S4_05_hybrid_rag.ipynb`